# 📥 Download and Filter NASA GeneLab Omics Datasets

This notebook automates the retrieval and pre‑processing of omics datasets from the NASA GeneLab Open Science Data Repository (OSDR) using the `genelab_utils` package. It supports both incremental and full updates, applies pre‑filters to reduce file size, and writes a manifest of downloaded files.

Author: Peter W. Rose, UC San Diego (pwrose.ucsd@gmail.com)

In [1]:
import os
import pandas as pd
import genelab_utils as gl

pd.set_option("display.max_rows", None)

In [2]:
os.makedirs("../data", exist_ok=True)

In [3]:
MANIFEST_PATH = "../data/manifest.csv" # file to save dataset info

## Incremental vs Full Update
By default, this notebook runs an incremental update. It downloads and preprocesses any new datasets specified in the "technology_types" list below.

If any datasets have been updated, set the "reset" variable to "True" to run a complete update.

The downloaded datasets are saved in the "datasets" directory.

In [4]:
RESET = False # run incremental update
# RESET = True # run a complete update to refresh datasets

## Get a List of GeneLab processed Datasets

In [5]:
dataset_info = gl.get_processed_datasets()
dataset_info[dataset_info["taxonomy"] == "6239"]

,identifier,technology,measurement,assay_name,taxonomy,material,host_organism,host_strain,filename,url,differential_analysis_method
367,OSD-42,DNA microarray,transcription profiling,OSD-42_transcription-profiling_dna-microarray_...,6239,,,,GLDS-42_array_differential_expression_GLmicroa...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2
1245,OSD-112,DNA microarray,transcription profiling,OSD-112_transcription-profiling_dna-microarray...,6239,Whole Organism,,,GLDS-112_array_differential_expression_GLmicro...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2
1254,OSD-113,DNA microarray,transcription profiling,OSD-113_transcription-profiling_dna-microarray,6239,Whole Organism,,,GLDS-113_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2
5769,OSD-425,DNA microarray,transcription profiling,OSD-425_transcription-profiling_dna-microarray...,6239,Whole Organism,,,GLDS-425_array_differential_expression_GLmicro...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2


## Filter by Technology Type

In [6]:
technology_types = ["RNA Sequencing (RNA-Seq)", 
                    "DNA microarray", 
                    "Whole Genome Bisulfite Sequencing",
                    "Reduced-Representation Bisulfite Sequencing",
                    "16S",
                    "ITS",
                    "18S",
                    "16S and ITS",
                   ]
dataset_info = gl.filter_by_technology_type(dataset_info, technology_types)

## Filter by Organism

In [7]:
print(f"Available organisms: {dataset_info['taxonomy'].dropna().unique()}")

Available organisms: ['7227' '10090' '10116' '9606' '' '287' '3702' '6239' '562' '4932' '1423'
 '13613' '7955' '63436' '63433']


In [8]:
taxids = {"9606": "Homo sapiens",
          # -- Rodens -- 
          "10090": "Mus musculus",
          "10116": "Rattus norvegicus",
          # -- Fish --
          "7955": "Danio rerio",
          "8090": "Oryzias latipes",
          # -- Nematoda --
          "6239": "Caenorhabditis elegans",
          # -- Insecta --
          "7227": "Drosophila melanogaster",
          "63436": "Leptopilina heterotoma",
          "63433": "Leptopilina boulardi",
          # -- Bacteria --
          "562": "Escherichia coli",
          "287": "Pseudomonas aeruginosa",
          "1423": "Bacillus subtilis",
          "1452": "Bacillus atrophaeus",
          "1781": "Mycobacterium marinum",
          "148447": "Paraburkholderia phymatum",
          # -- Fungi --
          "4932": "Saccharomyces cerevisiae",
          # -- Plants --
          "3711": "Brassica rapa",
          "15368": "Brachypodium distachyon",
          "3702": "Arabidopsis thaliana",
          # -- Other --
          "104782": "Adineta vaga",
          "13613": "Microbiota",
          "131567": "cellular organisms",
          # -- Host organisms
          "4236": "Lactuca sativa",
         }
          
dataset_info = gl.filter_by_organism(dataset_info, taxids)
dataset_info = gl.filter_by_host_organism(dataset_info, taxids)

In [9]:
print(f"Filtered organisms: {dataset_info['taxonomy'].unique()}")

Filtered organisms: ['7227' '10090' '10116' '9606' '287' '3702' '6239' '562' '4932' '1423'
 '13613' '7955' '63436' '63433']


In [10]:
dataset_info.head()

,identifier,technology,measurement,assay_name,taxonomy,material,host_organism,host_strain,filename,url,differential_analysis_method,organism,host_taxonomy
0,OSD-1,DNA microarray,transcription profiling,OSD-1_transcription-profiling_dna-microarray_A...,7227,Whole Organism,,,GLDS-1_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Drosophila melanogaster,
18,OSD-3,DNA microarray,transcription profiling,OSD-3_transcription-profiling_dna-microarray_a...,7227,Whole Organism,,,GLDS-3_array_differential_expression_GLmicroar...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Drosophila melanogaster,
36,OSD-4,DNA microarray,transcription profiling,OSD-4_transcription-profiling_dna-microarray_a...,10090,thymus,,,GLDS-4_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Mus musculus,
44,OSD-6,DNA microarray,transcription profiling,OSD-6_transcription-profiling_dna-microarray_a...,10116,Cells,,,GLDS-6_array_differential_expression_GLmicroar...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Rattus norvegicus,
50,OSD-13,DNA microarray,transcription profiling,OSD-13_transcription-profiling_dna-microarray_...,9606,Primary T Cells,,,GLDS-13_array_differential_expression_GLmicroa...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Homo sapiens,


## Select Datasets to Download
The map below specifies the technology type and a substring used to identify processed files. Processed files must contain this substring.

In [12]:
file_types = {"DNA microarray": "differential_expression",
              "RNA Sequencing (RNA-Seq)": "differential_expression",
              "Whole Genome Bisulfite Sequencing": "differential_methylation_tiles",
              "Reduced-Representation Bisulfite Sequencing": "differential_methylation_tiles",
              "16S": "differential_abundance",
              "ITS": "differential_abundance",
              "18S": "differential_abundance",
              "16S and ITS": "differential_abundance",
              }

#### Define pre-filters to reduce the file the essential data

In [13]:
def differential_expression_filter(df, threshold=0.1):
    if "ENTREZID" not in df.columns:
        return pd.DataFrame()
        
    filtered_df = df[df['ENTREZID'].notna() & (df['ENTREZID'].astype(str) != '')]
    # Keep only required columns
    filtered_df = filtered_df.filter(regex=r"^(ENTREZID|SYMBOL|GENENAME|Log2fc_|Adj\.p\.value_|Group\.Mean_|Group\.Stdev_)")
    adj_pval_cols = [col for col in filtered_df.columns if col.startswith("Adj.p.value_")]
    filtered_df = filtered_df[filtered_df[adj_pval_cols].le(threshold).any(axis=1)]
    # Explode rows with multiple genes
    if "ENTREZID" in filtered_df.columns:
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].astype(str)
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].apply(lambda x:x.split('|'))
        filtered_df = filtered_df.explode('ENTREZID')
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].str.strip()
    return filtered_df

In [14]:
def differential_methylation_filter(df, threshold=0.1):
    # add reversed contrast:
    #    meth.diff_(Ground Control)v(Space Flight) -> meth.diff_(Space Flight)v(Ground Control) and reverse sign
    #    qvalue_(Ground Control)v(Space Flight) -> qvalue_(Space Flight)v(Ground Control)
    df = gl.add_reversed_contrast_columns(df)
    
    filtered_df = df[df['ENTREZID'].notna() & (df['ENTREZID'].astype(str) != '')]
    # Keep only required columns
    filtered_df = filtered_df.filter(regex=r"^(ENTREZID|SYMBOL|GENENAME|chr|start|end|dist.to.feature|prom|exon|intron|meth.diff_|qvalue_|Group\.Mean_|Group\.Stdev_)")
    qval_cols = [col for col in filtered_df.columns if col.startswith("qvalue_")]
    filtered_df = filtered_df[filtered_df[qval_cols].le(threshold).any(axis=1)]
     # Explode rows with multiple genes
    if "ENTREZID" in filtered_df.columns:
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].astype(str)
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].apply(lambda x:x.split('|'))
        filtered_df = filtered_df.explode('ENTREZID')
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].str.strip()
    return filtered_df

In [15]:
def differential_abundance_filter(df, threshold=0.1):
    s = df["NCBI_id"]

    mask = (
            s.notna()                               # real NaNs
            & s.astype(str).str.strip().ne("")      # empty/whitespace
            & s.astype(str).str.strip().str.lower().ne("nan")  # literal "nan"
    )        

    filtered_df = df[mask].copy()
    filtered_df["NCBI_id"] = filtered_df["NCBI_id"].astype(int)

    # Keep only required columns
    filtered_df = filtered_df.filter(regex=r"^(best_taxonomy|NCBI_id|Lnfc_|Log2fc_|Q\.value_|Adj\.p\.value_|Group\.Mean_|Group\.Stdev_)")
    adj_pval_cols = [col for col in filtered_df.columns if col.startswith("Adj.p.value_")]
    if len(adj_pval_cols) > 0:
        filtered_df = filtered_df[filtered_df[adj_pval_cols].le(threshold).any(axis=1)]
    qval_cols = [col for col in filtered_df.columns if col.startswith("Q.value_")]
    if len(qval_cols) > 0:
        filtered_df = filtered_df[filtered_df[qval_cols].le(threshold).any(axis=1)]

    return filtered_df

In [16]:
filters = {"differential_expression": differential_expression_filter,
           "differential_methylation_tiles": differential_methylation_filter,
           "differential_abundance": differential_abundance_filter,}

In [17]:
manifest = gl.download_data_files(dataset_info, file_types, filters, reset=RESET)
manifest.to_csv(MANIFEST_PATH, index=False)

File already exist: GLDS-1_array_differential_expression.csv
File already exist: GLDS-3_array_differential_expression_GLmicroarray.csv
File already exist: GLDS-4_array_differential_expression.csv
File already exist: GLDS-6_array_differential_expression_GLmicroarray.csv
File already exist: GLDS-13_array_differential_expression_GLmicroarray.csv
Downloading: GLDS-15_array_differential_expression_GLmicroarray.csv
Skipping file: GLDS-15_array_differential_expression_GLmicroarray.csv. No data after filtering.
File already exist: GLDS-22_array_differential_expression.csv
File already exist: GLDS-25_array_differential_expression_GLmicroarray.csv
File already exist: GLDS-27_array_differential_expression_GLmicroarray.csv
Downloading: GLDS-30_array_differential_expression_GLmicroarray.csv
Skipping file: GLDS-30_array_differential_expression_GLmicroarray.csv. No data after filtering.
File already exist: GLDS-34_array_differential_expression_GLmicroarray.csv
File already exist: GLDS-36_array_differ

In [18]:
manifest.head()

,identifier,technology,measurement,assay_name,taxonomy,material,host_organism,host_strain,filename,url,differential_analysis_method,organism,host_taxonomy
0,OSD-1,DNA microarray,transcription profiling,OSD-1_transcription-profiling_dna-microarray_A...,7227,Whole Organism,,,GLDS-1_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Drosophila melanogaster,
18,OSD-3,DNA microarray,transcription profiling,OSD-3_transcription-profiling_dna-microarray_a...,7227,Whole Organism,,,GLDS-3_array_differential_expression_GLmicroar...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Drosophila melanogaster,
36,OSD-4,DNA microarray,transcription profiling,OSD-4_transcription-profiling_dna-microarray_a...,10090,thymus,,,GLDS-4_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Mus musculus,
44,OSD-6,DNA microarray,transcription profiling,OSD-6_transcription-profiling_dna-microarray_a...,10116,Cells,,,GLDS-6_array_differential_expression_GLmicroar...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Rattus norvegicus,
50,OSD-13,DNA microarray,transcription profiling,OSD-13_transcription-profiling_dna-microarray_...,9606,Primary T Cells,,,GLDS-13_array_differential_expression_GLmicroa...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2,Homo sapiens,
